In [ ]:
import re, requests, pandas as pd
from bs4 import BeautifulSoup
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import NMF

urls = ['https://www.bbc.com/news/technology', 'https://www.cnn.com/tech', 'https://en.wikipedia.org/wiki/Data_mining']

def scrape(url):
    try:
        r = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=15); r.raise_for_status()
        return ' '.join(p.get_text(' ', strip=True) for p in BeautifulSoup(r.text, 'html.parser').find_all('p'))
    except Exception as e:
        print('Could not access', url, e); return ''

docs = [t for t in map(scrape, urls) if t.strip()]
print('Pages scraped:', len(docs))

In [ ]:
clean = [' '.join(w for w in re.findall(r'[a-z]+', t.lower()) if w not in ENGLISH_STOP_WORDS and len(w) > 2) for t in docs]
vec = TfidfVectorizer()
tfidf = vec.fit_transform(clean)
terms = vec.get_feature_names_out()

In [ ]:
nmf = NMF(n_components=min(3, tfidf.shape[0]), random_state=42, max_iter=500).fit(tfidf)
for i, topic in enumerate(nmf.components_):
    print(f'Topic {i+1}: ' + ', '.join(terms[topic.argsort()[-10:][::-1]]))

trend = pd.Series(nmf.components_.sum(axis=0), index=terms).sort_values(ascending=False)
print('\nTrending terms:\n', trend.head(10))